# centaur-lab data explorer

All connection plumbing lives in the `centaur_db` package — `db.connect()` starts a kubectl port-forward to the in-cluster Postgres if one isn't already up, and tears it down on kernel shutdown.

Use `db.query(conn, sql, params)` for SELECTs (returns a `pandas.DataFrame`) and `db.execute(conn, sql, params)` for INSERT/UPDATE/DELETE (returns rowcount).

In [ ]:
import centaur_db as db

conn = db.connect()
q = lambda sql, params=None: db.query(conn, sql, params)
q("SELECT current_database() AS db, version() AS pg_version").iloc[0].to_dict()

## #local-protocol — recent messages and projected docs

In [ ]:
channel = q(
    "SELECT channel_id, channel_name FROM slack_sync_channels WHERE channel_name = %s",
    ("local-protocol",),
).iloc[0]
channel_id = channel.channel_id

messages = q(
    "SELECT occurred_at, user_id, reply_count, left(text, 240) AS preview "
    "FROM slack_sync_messages WHERE channel_id = %s ORDER BY occurred_at DESC",
    (channel_id,),
)
docs = q(
    "SELECT source_type, title, length(body) AS body_len, source_updated_at "
    "FROM company_context_documents WHERE source = 'slack' AND source_document_id LIKE %s "
    "ORDER BY source_updated_at DESC",
    (f"%{channel_id}%",),
)
print(f"#{channel.channel_name}: {len(messages)} messages, {len(docs)} projected docs")
messages.head(10)

## BM25 search via ParadeDB

Centaur's Postgres is the [ParadeDB](https://docs.paradedb.com) image (`paradedb/paradedb:0.23.0-pg16`), with a `pg_search` BM25 index on `company_context_documents`. `body @@@ 'query'` is the BM25 match operator; `paradedb.score(document_id)` returns the relevance score. This is what the agent's company-context retriever uses under the hood.

In [ ]:
q(
    "SELECT source_type, title, paradedb.score(document_id) AS score, "
    "       left(body, 240) AS preview "
    "FROM company_context_documents "
    "WHERE body @@@ %s "
    "ORDER BY score DESC LIMIT 5",
    ("slack backfill",),
)